### **Modelo causal decoder-only pequeño**

El flujo conceptual completo de este cuaderno es:

$$
\text{corpus} \rightarrow \text{tokenización} \rightarrow \text{secuencias discretas}
\rightarrow \text{embeddings} \rightarrow \text{Transformer causal}
\rightarrow \text{distribución sobre el siguiente token}
\rightarrow \text{decoding} \rightarrow \text{texto generado}.
$$

A lo largo del cuaderno se insistirá en cinco ideas:

1. el modelo no opera sobre palabras, sino sobre **tokens**,
2. el entrenamiento causal implementa la factorización autoregresiva,
3. la calidad de generación depende de **datos, capacidad y convergencia**, no solo de la política de decoding,
4. la ventana de contexto y el costo de inferencia están ligados a la longitud en tokens,
5. comparar con un modelo preentrenado permite distinguir entre **arquitectura**, **entrenamiento** y **escala**.


#### **1. Instalación opcional**

Descomenta esta celda si estás en Colab o en un entorno limpio.

En un entorno de trabajo más formal, convendría fijar versiones exactas para garantizar reproducibilidad experimental.


In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# !pip install -q transformers datasets matplotlib

#### **2. Importación de librerías y configuración**

Aquí importamos las librerías necesarias para:

- cargar el corpus,
- tokenizar texto,
- definir el modelo,
- optimizar parámetros,
- visualizar métricas.

Desde una perspectiva metodológica, esto separa cuatro capas:

$$
\text{datos} \rightarrow \text{representación} \rightarrow \text{modelo} \rightarrow \text{evaluación}.
$$

Esa separación es útil porque permite diagnosticar con más precisión dónde falla un sistema: en el corpus, en el preprocesamiento, en la arquitectura o en la inferencia.


In [ ]:
import math
import random
from collections import Counter
from typing import List

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import pandas as pd
import matplotlib.pyplot as plt

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed

set_seed(42)
random.seed(42)
torch.manual_seed(42)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo:", DEVICE)


#### **3. Tokenizador de subpalabras**

Usaremos el tokenizador de `distilgpt2`.

Formalmente, un tokenizador puede verse como una aplicación

$$
\tau: \Sigma^\* \to \mathcal V^\*,
$$

donde:

- $\Sigma^*$ es el conjunto de cadenas posibles,
- $\mathcal V$ es el vocabulario discreto del modelo,
- y $\tau(s)$ convierte un texto $s$ en una secuencia de tokens.

Esto es importante porque el costo computacional del modelo no depende del número de palabras ortográficas, sino de la longitud de la secuencia tokenizada:

$$
T = |\tau(s)|.
$$

En LLMs modernos, trabajar con **subpalabras** permite equilibrar:
- cobertura léxica,
- tamaño del vocabulario,
- robustez frente a palabras raras o morfología productiva.


In [ ]:
MODEL_TOKENIZER = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_TOKENIZER)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Vocab size:", tokenizer.vocab_size)
print("Pad token:", tokenizer.pad_token, tokenizer.pad_token_id)
print("EOS token:", tokenizer.eos_token, tokenizer.eos_token_id)


#### **4. Carga del corpus**

Usamos una porción de IMDB como colección de secuencias textuales.

Sea el corpus

$$
\mathcal D = \{ s^{(i)} \}_{i=1}^{N}, \qquad s^{(i)} \in \Sigma^*.
$$

Aunque IMDB suele utilizarse para clasificación de sentimiento, aquí ignoramos las etiquetas y retenemos solo el texto.  
Eso basta para construir un problema de modelado del lenguaje, porque lo único que necesitamos es una familia de secuencias sobre las que entrenar la predicción del siguiente token.


In [ ]:
dataset = load_dataset("imdb")

train_texts = dataset["train"]["text"][:2000]
valid_texts = dataset["test"]["text"][:400]

print("Ejemplos de entrenamiento:", len(train_texts))
print("Ejemplos de validación:", len(valid_texts))
print("\nPrimer ejemplo:\n")
print(train_texts[0][:700], "...")


#### **5. Tokenización del corpus**

Cada documento se tokeniza y luego se concatena usando `eos_token_id` como separador.

Si cada documento produce una secuencia tokenizada

$$
\tau(s^{(i)}) = x^{(i)}_{1:T_i},
$$

la concatenación global se puede pensar como

$$
\widetilde{x}_{1:M}
=
x^{(1)}_{1:T_1}\oplus \langle eos\rangle \oplus x^{(2)}_{1:T_2}\oplus \cdots
$$

donde $\oplus$ denota concatenación.

Esta decisión simplifica mucho la construcción del dataset causal, aunque también introduce una limitación: el modelo puede ver como adyacentes el final de una reseña y el comienzo de la siguiente.


In [ ]:
def tokenize_corpus(texts: List[str], max_docs: int = None) -> List[int]:
    ids = []
    subset = texts if max_docs is None else texts[:max_docs]
    for txt in subset:
        toks = tokenizer.encode(txt, add_special_tokens=False)
        ids.extend(toks + [tokenizer.eos_token_id])
    return ids

train_ids = tokenize_corpus(train_texts)
valid_ids = tokenize_corpus(valid_texts)

print("Total tokens train:", len(train_ids))
print("Total tokens valid:", len(valid_ids))
print("Primeros 30 ids:", train_ids[:30])


#### **6. Diagnóstico de longitudes del corpus**

Esta sección permite interpretar advertencias del tokenizer relacionadas con secuencias largas.

Dado un texto $s$, la longitud relevante es:

$$
T = |\tau(s)|.
$$

En modelos de referencia como GPT-2 o DistilGPT-2, existe un límite máximo nominal de posiciones.  
Si algunos documentos tienen $T > L_{\max}$, eso no implica necesariamente que el experimento falle, pero sí exige distinguir entre:

- longitud original del documento,
- longitud del bloque efectivamente usado para entrenamiento.


In [ ]:
lengths = [len(tokenizer.encode(t, add_special_tokens=False)) for t in train_texts[:500]]

print("Máximo:", max(lengths))
print("Promedio:", sum(lengths) / len(lengths))
print("Mayores a 1024:", sum(l > 1024 for l in lengths))


#### **7. Nota sobre longitudes**

Algunas reseñas pueden exceder la longitud máxima nominal del tokenizer base.

Sin embargo, en este cuaderno entrenamos sobre bloques de tamaño fijo `SEQ_LEN`, de modo que la secuencia global se fragmenta en subsecuencias cortas.  
La advertencia del tokenizer es útil porque recuerda una restricción del modelo de referencia, pero no invalida automáticamente el experimento local.

La lección metodológica es importante: una advertencia sobre longitud debe interpretarse dentro del pipeline completo y no de manera aislada.


#### **8. Dataset causal**

Construimos pares $(x,y)$ donde $y$ es la versión desplazada de $x$.

Si

$$
x = (x_1,\dots,x_T),
$$

entonces el objetivo es,

$$
y = (x_2,\dots,x_{T+1}).
$$

Esto implementa la factorización autoregresiva,

$$
p_\theta(x_{1:T}) = \prod_{t=1}^{T} p_\theta(x_t \mid x_{<t}),
$$

porque cada posición aprende a predecir el siguiente token condicionado en el prefijo.


In [ ]:
class CausalLMDataset(Dataset):
    def __init__(self, token_ids: List[int], seq_len: int = 64, stride: int = 64):
        self.examples = []

        for start in range(0, len(token_ids) - seq_len - 1, stride):
            x = token_ids[start:start + seq_len]
            y = token_ids[start + 1:start + seq_len + 1]
            self.examples.append((
                torch.tensor(x, dtype=torch.long),
                torch.tensor(y, dtype=torch.long)
            ))

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx]

SEQ_LEN = 64
BATCH_SIZE = 32

train_ds = CausalLMDataset(train_ids, seq_len=SEQ_LEN, stride=SEQ_LEN)
valid_ds = CausalLMDataset(valid_ids, seq_len=SEQ_LEN, stride=SEQ_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE)

print("Batches train:", len(train_loader))
print("Batches valid:", len(valid_loader))


#### **9. Verificación de un lote**

Antes de entrenar, conviene verificar que el dataset causal esté bien construido.

La condición deseada es:

$$
y_t = x_{t+1}.
$$

Esta comprobación es esencial, porque un error silencioso en la construcción del dataset puede hacer inútil todo el entrenamiento, incluso si el código del optimizador y del modelo es correcto.


In [ ]:
x_batch, y_batch = next(iter(train_loader))
print("Forma de x:", x_batch.shape)
print("Forma de y:", y_batch.shape)

print("\nPrimer ejemplo x (ids):")
print(x_batch[0][:20])

print("\nPrimer ejemplo y (ids):")
print(y_batch[0][:20])

print("\nPrimer ejemplo x decodificado:")
print(tokenizer.decode(x_batch[0].tolist(), clean_up_tokenization_spaces=False))


#### **10. Codificación posicional**

La autoatención, por sí sola, no codifica orden.  
Por eso añadimos una señal posicional sinusoidal.

Para posición $pos$ y dimensión $i$, la codificación clásica toma la forma:

$$
PE_{pos,2i} = \sin\left(\frac{pos}{10000^{2i/d}}\right),
\qquad
PE_{pos,2i+1} = \cos\left(\frac{pos}{10000^{2i/d}}\right).
$$

Si $E[x_t]$ es el embedding del token, la entrada efectiva al Transformer es:

$$
h_t^{(0)} = E[x_t] + PE_t.
$$

Esto dota al modelo de información de orden sin necesidad de recurrencia explícita.


In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, dropout: float = 0.1, max_len: int = 2048):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)

        self.register_buffer("pe", pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


#### **11. Modelo decoder-only pequeño**

Se define un modelo tipo GPT pequeño usando `nn.TransformerEncoder` con máscara causal.

Aunque el nombre de la clase en PyTorch sea "encoder", al imponer una máscara triangular superior obtenemos un comportamiento autoregresivo.  
La máscara causal puede describirse idealmente como:

$$
M_{ij} =
\begin{cases}
0 & \text{si } j \le i,\\
-\infty & \text{si } j > i.
\end{cases}
$$

Eso garantiza que la posición $i$ solo atienda al contexto izquierdo.

En términos abstractos, el modelo implementa:

$$
x_{1:T}
\rightarrow
H^{(0)}
\rightarrow
H^{(1)}
\rightarrow \cdots \rightarrow
H^{(L)}
\rightarrow
Z,
$$

donde $Z \in \mathbb{R}^{T \times |\mathcal V|}$ son los logits sobre el vocabulario.


In [ ]:
class TinyGPT(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        d_model: int = 256,
        n_heads: int = 4,
        n_layers: int = 4,
        d_ff: int = 512,
        dropout: float = 0.1,
        max_len: int = 2048
    ):
        super().__init__()
        self.d_model = d_model
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_enc = PositionalEncoding(d_model, dropout=dropout, max_len=max_len)

        layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_ff,
            dropout=dropout,
            batch_first=True,
            activation="gelu"
        )
        self.transformer = nn.TransformerEncoder(layer, num_layers=n_layers)
        self.lm_head = nn.Linear(d_model, vocab_size)

        def _reset_parameters(module):
            pass

    def causal_mask(self, T: int, device: str):
        return torch.triu(torch.ones(T, T, device=device, dtype=torch.bool), diagonal=1)

    def forward(self, input_ids):
        x = self.token_emb(input_ids) * math.sqrt(self.d_model)
        x = self.pos_enc(x)
        T = input_ids.size(1)
        mask = self.causal_mask(T, input_ids.device)
        h = self.transformer(x, mask=mask)
        logits = self.lm_head(h)
        return logits

VOCAB_SIZE = tokenizer.vocab_size

model = TinyGPT(
    vocab_size=VOCAB_SIZE,
    d_model=256,
    n_heads=4,
    n_layers=4,
    d_ff=512,
    dropout=0.1,
    max_len=SEQ_LEN
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters())
print(f"Parámetros del modelo: {n_params:,}")


#### **12. Función de pérdida y optimización**

Usamos entropía cruzada por token.  
Si $z_t$ son los logits en la posición $t$, la distribución predicha es:

$$
p_\theta(v \mid x_{<t}) =
\frac{\exp(z_{t,v})}{\sum_{u \in \mathcal V}\exp(z_{t,u})}.
$$

La pérdida promedio puede escribirse como:

$$
\mathcal L(\theta)
=
-\frac{1}{T}\sum_{t=1}^{T}\log p_\theta(x_t \mid x_{<t}).
$$

Esto equivale a máxima verosimilitud autoregresiva sobre el corpus.

Como optimizador usamos AdamW, una variante robusta para entrenamiento de Transformers pequeños y medianos.


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

def run_epoch(model, loader, optimizer=None):
    train = optimizer is not None
    model.train() if train else model.eval()

    total_loss = 0.0
    total_tokens = 0

    for x, y in loader:
        x = x.to(DEVICE)
        y = y.to(DEVICE)

        with torch.set_grad_enabled(train):
            logits = model(x)
            loss = criterion(logits.reshape(-1, logits.size(-1)), y.reshape(-1))

            if train:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

        total_loss += loss.item() * y.numel()
        total_tokens += y.numel()

    avg_loss = total_loss / total_tokens
    ppl = math.exp(avg_loss) if avg_loss < 20 else float("inf")
    return avg_loss, ppl


#### **13. Entrenamiento**

Este entrenamiento es didáctico.  
No pretende competir con un LLM industrial, sino mostrar que el pipeline causal es operativo.

En general, si aumentan:
- los datos,
- el tamaño del modelo,
- y el cómputo,

la calidad del modelado mejora.  
Pero aquí trabajamos deliberadamente en un régimen pequeño, para que el comportamiento del sistema siga siendo observable y modificable en este cuaderno.


In [ ]:
EPOCHS = 3

history = []
for epoch in range(1, EPOCHS + 1):
    train_loss, train_ppl = run_epoch(model, train_loader, optimizer=optimizer)
    valid_loss, valid_ppl = run_epoch(model, valid_loader, optimizer=None)

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "train_ppl": train_ppl,
        "valid_loss": valid_loss,
        "valid_ppl": valid_ppl
    }
    history.append(row)
    print(row)

history_df = pd.DataFrame(history)
history_df


#### **14. Lectura de métricas finales**

Antes de interpretar la generación, conviene mirar `loss` y `perplexity`.

La **perpĺejidad (perplexity)** se define como:

$$
\mathrm{PPL} = \exp(\mathcal L).
$$

Puede leerse como una medida exponencial del error medio por token.  
No es una métrica perfecta de calidad textual, pero sí un buen diagnóstico preliminar del estado del entrenamiento.


In [ ]:
print("Última época:")
print(history_df.tail(1))

best_valid = history_df["valid_loss"].min()
best_ppl = history_df["valid_ppl"].min()

print(f"Mejor valid_loss: {best_valid:.4f}")
print(f"Mejor valid_ppl : {best_ppl:.2f}")


#### **15. Cómo interpretar loss y perplexity**

Si `valid_loss` y `valid_ppl` siguen siendo altas, no debe esperarse texto fluido ni estable.

Este punto es crucial: la calidad generativa no depende solo del decoding.  
Antes de culpar a `top-k` o `top-p`, conviene verificar si el modelo realmente ha aprendido una distribución útil sobre el corpus.

En otras palabras, si

$$
\mathcal L \text{ sigue alta},
$$

entonces el sistema aún no aproxima bien

$$
p_\theta(x_t \mid x_{<t}),
$$

y por tanto cualquier política de generación operará sobre una base débil.


#### **16. Curvas de entrenamiento**

La visualización ayuda a detectar:
- descenso estable,
- estancamiento,
- separación excesiva entre entrenamiento y validación.

En una lectura más rigurosa, estas curvas deben interpretarse junto con:
- ejemplos generados,
- dominio del corpus,
- y tamaño del modelo.


In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history_df["epoch"], history_df["train_loss"], marker="o", label="train_loss")
plt.plot(history_df["epoch"], history_df["valid_loss"], marker="o", label="valid_loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Curvas de entrenamiento")
plt.legend()
plt.show()


#### **17. Nota sobre expectativas de generación**

El modelo entrenado aquí es pequeño, se entrena desde cero y usa una muestra reducida de IMDB en inglés.

Por tanto:

- no debe compararse con GPT-2 real ni con asistentes conversacionales modernos,
- greedy puede producir repeticiones,
- sampling puede producir secuencias incoherentes,
- los prompts fuera de dominio o fuera de idioma pueden degradar fuertemente la salida.

Esta aclaración es importante porque permite separar tres fenómenos:

$$
\text{capacidad del modelo},
\quad
\text{calidad del entrenamiento},
\quad
\text{desajuste de dominio}.
$$


#### **18. Función de generación básica**

En inferencia autoregresiva, dado un contexto $x_{1:t}$, el modelo produce una distribución sobre el siguiente token:

$$
p_\theta(x_{t+1}\mid x_{1:t}) = \mathrm{softmax}(z_t).
$$

Luego una política de decodificación elige $\hat{x}_{t+1}$, se concatena al contexto y se repite el proceso.

Esta función permite:
- **greedy**, donde se elige el máximo,
- **sampling**, donde se muestrea desde la distribución,
- y variantes restringidas mediante `top_k` y `top_p`.


In [ ]:
@torch.no_grad()
def generate_text(model, tokenizer, prompt: str, max_new_tokens: int = 40, do_sample: bool = False,
                  temperature: float = 1.0, top_k: int = 0, top_p: float = 1.0):
    model.eval()
    input_ids = tokenizer.encode(prompt, add_special_tokens=False, return_tensors="pt").to(DEVICE)

    for _ in range(max_new_tokens):
        idx_cond = input_ids[:, -SEQ_LEN:]
        logits = model(idx_cond)
        next_token_logits = logits[:, -1, :]

        if do_sample and temperature != 1.0:
            next_token_logits = next_token_logits / max(temperature, 1e-6)

        probs = torch.softmax(next_token_logits, dim=-1)

        if do_sample:
            if top_k and top_k > 0:
                v, _ = torch.topk(probs, top_k)
                min_keep = v[:, -1].unsqueeze(-1)
                probs = torch.where(probs < min_keep, torch.zeros_like(probs), probs)
                probs = probs / probs.sum(dim=-1, keepdim=True)

            if top_p < 1.0:
                sorted_probs, sorted_idx = torch.sort(probs, descending=True)
                cumulative = torch.cumsum(sorted_probs, dim=-1)
                mask = cumulative > top_p
                mask[:, 1:] = mask[:, :-1].clone()
                mask[:, 0] = False
                sorted_probs[mask] = 0
                sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
                next_id = torch.multinomial(sorted_probs, num_samples=1)
                next_token = torch.gather(sorted_idx, -1, next_id)
            else:
                next_token = torch.multinomial(probs, num_samples=1)
        else:
            next_token = torch.argmax(probs, dim=-1, keepdim=True)

        input_ids = torch.cat([input_ids, next_token], dim=1)

    return tokenizer.decode(input_ids[0].tolist(), clean_up_tokenization_spaces=False)


#### **19. Función de generación más conservadora**

La política `generate_safe` intenta reducir repeticiones y ruido.

Matemáticamente, las estrategias de decodificación modifican la distribución efectiva.  
Por ejemplo, con temperatura $\tau$:

$$
p^{(\tau)}(v\mid x_{1:t})
=
\frac{\exp(z_v/\tau)}{\sum_u \exp(z_u/\tau)}.
$$

- Si $\tau < 1$, la distribución se vuelve más concentrada.
- Si $\tau > 1$, se vuelve más dispersa.

Top-k y top-p restringen el soporte de muestreo, y la penalización por repetición deforma la distribución para evitar bucles locales.

Eso no "arregla" un modelo débil, pero sí modula cómo se explotan sus probabilidades.


In [ ]:
@torch.no_grad()
def generate_safe(model, tokenizer, prompt: str, max_new_tokens: int = 40, temperature: float = 0.7,
                  top_k: int = 20, top_p: float = 0.9, repetition_penalty: float = 1.15):
    model.eval()
    input_ids = tokenizer.encode(prompt, add_special_tokens=False, return_tensors="pt").to(DEVICE)

    for _ in range(max_new_tokens):
        idx_cond = input_ids[:, -SEQ_LEN:]
        logits = model(idx_cond)
        next_token_logits = logits[:, -1, :]

        for token_id in set(input_ids[0].tolist()):
            next_token_logits[0, token_id] /= repetition_penalty

        next_token_logits = next_token_logits / max(temperature, 1e-6)
        probs = torch.softmax(next_token_logits, dim=-1)

        v, _ = torch.topk(probs, top_k)
        min_keep = v[:, -1].unsqueeze(-1)
        probs = torch.where(probs < min_keep, torch.zeros_like(probs), probs)

        sorted_probs, sorted_idx = torch.sort(probs, descending=True)
        cumulative = torch.cumsum(sorted_probs, dim=-1)
        mask = cumulative > top_p
        mask[:, 1:] = mask[:, :-1].clone()
        mask[:, 0] = False
        sorted_probs[mask] = 0
        sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)

        next_id = torch.multinomial(sorted_probs, num_samples=1)
        next_token = torch.gather(sorted_idx, -1, next_id)

        input_ids = torch.cat([input_ids, next_token], dim=1)

    return tokenizer.decode(input_ids[0].tolist(), clean_up_tokenization_spaces=False)


#### **20. Prueba en dominio del corpus**

Antes de usar prompts abiertos o en otro idioma, conviene probar el modelo con prompts en inglés y cercanos al dominio de IMDB.

Este paso ayuda a aislar la variable "desajuste de dominio".  
Si el modelo fue entrenado sobre reseñas en inglés, es razonable esperar mejor comportamiento cuando el prompt pertenece al mismo universo discursivo.


In [ ]:
prompts_dominio = [
    "This movie is",
    "The plot of the film",
    "The acting was",
    "I think this movie"
]

for p in prompts_dominio:
    print("PROMPT:", p)
    print(generate_text(model, tokenizer, p, max_new_tokens=40, do_sample=False))
    print()


#### **21. Prueba fuera de dominio/fuera de idioma**

Este prompt está en español, mientras que el modelo fue entrenado con reseñas IMDB en inglés.

La finalidad de esta prueba no es "medir calidad máxima", sino mostrar cómo el rendimiento puede degradarse cuando la distribución del prompt difiere de la del corpus de entrenamiento.

En términos estadísticos, es una forma sencilla de observar un problema de **shift de dominio**.


In [ ]:
prompt = "La inteligencia artificial"

print("Greedy")
print(generate_text(model, tokenizer, prompt, max_new_tokens=40, do_sample=False))

print("\nSampling T=0.8")
print(generate_text(model, tokenizer, prompt, max_new_tokens=40, do_sample=True, temperature=0.8))

print("\nTop-k=40")
print(generate_text(model, tokenizer, prompt, max_new_tokens=40, do_sample=True, top_k=40))

print("\nTop-p=0.9")
print(generate_text(model, tokenizer, prompt, max_new_tokens=40, do_sample=True, top_p=0.9))


#### **22. Prueba con generación más estable**

Aquí se prueba `generate_safe` sobre un prompt en dominio.

La comparación entre `generate_text` y `generate_safe` permite observar una idea importante:  
el decoding no cambia los parámetros del modelo, solo cambia cómo se explota la distribución que el modelo ya aprendió.


In [ ]:
prompt = "The movie was"
print(generate_safe(model, tokenizer, prompt, max_new_tokens=40))


#### **23. Comparación con un modelo preentrenado**

Esta comparación es probablemente la parte más importante del diagnóstico.

La intención es distinguir entre:
- la validez del pipeline,
- la escala limitada del entrenamiento,
- y el efecto de usar un modelo ya preentrenado sobre una enorme cantidad de datos.

En otras palabras, queremos separar:

$$
\text{arquitectura correcta}
\neq
\text{modelo suficientemente entrenado}.
$$

Un modelo pequeño entrenado desde cero puede implementar bien la mecánica causal y, aun así, generar texto muy pobre.


In [ ]:
ref_name = "distilgpt2"
ref_tokenizer = AutoTokenizer.from_pretrained(ref_name)

if ref_tokenizer.pad_token is None:
    ref_tokenizer.pad_token = ref_tokenizer.eos_token

ref_model = AutoModelForCausalLM.from_pretrained(ref_name).to(DEVICE)
ref_model.eval()

@torch.no_grad()
def generate_hf(model, tokenizer, prompt, max_new_tokens=40, do_sample=True, temperature=0.8, top_p=0.9):
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        temperature=temperature,
        top_p=top_p,
        pad_token_id=tokenizer.eos_token_id
    )
    return tokenizer.decode(out[0], skip_special_tokens=True, clean_up_tokenization_spaces=False)


In [ ]:
prompt = "The movie was"

print("TinyGPT")
print(generate_text(model, tokenizer, prompt, max_new_tokens=40, do_sample=False))

print("\ndistilgpt2 preentrenado")
print(generate_hf(ref_model, ref_tokenizer, prompt, max_new_tokens=40))


#### **24. Diagnóstico de repetición**

Greedy puede colapsar en patrones repetitivos.  
Una forma simple de objetivar esto es medir repetición de n-gramas.

Si una secuencia contiene muchas repeticiones locales, eso sugiere que la distribución aprendida tiene picos demasiado dominantes o que el modelo aún no representa suficientemente bien transiciones variadas.


In [ ]:
def repetition_stats(text, n=3):
    words = text.split()
    ngrams = [" ".join(words[i:i+n]) for i in range(len(words)-n+1)]
    counts = Counter(ngrams)
    return {k: v for k, v in counts.items() if v > 1}

txt = generate_text(model, tokenizer, "The movie was", max_new_tokens=60, do_sample=False)
print(txt)
print("\nN-gramas repetidos:")
print(repetition_stats(txt, n=3))


#### **25. Comparación final de escenarios**

Se comparan:
- prompts en dominio,
- prompts abiertos,
- prompts fuera de idioma.

Esta sección ayuda a distinguir entre tres causas distintas de mala generación:

$$
\text{modelo pequeño}
\quad+\quad
\text{entrenamiento limitado}
\quad+\quad
\text{prompt fuera de distribución}.
$$

La gracia del análisis no está solo en "ver texto raro", sino en identificar de dónde proviene.


In [ ]:
prompts = [
    "The movie was",
    "This film is about",
    "La inteligencia artificial"
]

for p in prompts:
    print("PROMPT:", p)

    print("\n[Greedy]")
    print(generate_text(model, tokenizer, p, max_new_tokens=40, do_sample=False))

    print("\n[Safe decoding]")
    print(generate_safe(model, tokenizer, p, max_new_tokens=40))

    print()


#### **26. Ejercicios propuestos**

1. Cambia `SEQ_LEN` y discute el efecto sobre memoria, costo y ventana efectiva de contexto.
2. Entrena con menos ejemplos y compara `valid_loss` y `valid_ppl`.
3. Compara prompts en inglés del dominio IMDB frente a prompts en español.
4. Repite la comparación entre `TinyGPT` y `distilgpt2` con otros prompts.
5. Explica por qué `generate_safe` no arregla un modelo débil, sino que solo modifica la política de decodificación.
6. Discute qué parte del problema observable se debe a:
   - tamaño del modelo,
   - datos,
   - número de épocas,
   - y desajuste de dominio.

Preguntas más formales:
- ¿Qué limitaciones tiene la perplexity como métrica única de calidad?
- ¿Por qué una mejora local en $p_\theta(x_{t+1}\mid x_{1:t})$ no garantiza coherencia global?
- ¿Qué cambia cuando el prompt y el corpus pertenecen a distribuciones distintas?.
